# Sprite Mesh Refinery — Sierpiński Architecture

**Same principle as the Erdős sieve: cheap filters first, expensive filters last, each stage reduces what the next stage processes.**

**Validated by Pass 2 sieve results** — the elimination hierarchy (161/85/10/1 distribution) proves the same principle at 10^17 scale: most candidates fall to the cheapest filter, only survivors see the expensive ones. The Sierpiński architecture IS the sieve architecture.

**Pangea-Earth — same fracture pattern across all coordinates.** The sprite pipeline, the prime sieve, the game worlds — same recursive filter-then-refine at every scale. Zoom in, same structure. Different data, identical topology.

Pipeline hierarchy (cheapest → most expensive):
1. **Crop** — remove known chrome borders (free, eliminates 10-20% of pixels)
2. **Threshold** — alpha/luminance mask at thumbnail scale (milliseconds, identifies where the subject IS)
3. **Bounding box** — crop to subject region (reduces input to rembg by 40-70%)
4. **rembg** — neural network background removal on SMALLEST possible input
5. **MiDaS depth** — coarse → medium → fine (3 scales, each refines the last)
6. **Adaptive mesh** — subdivide only where depth gradient is high (flat areas stay coarse)

The Sierpiński triangle: same algorithm at every scale. Zoom in, same structure.
The sprite pipeline: same filter-then-refine at every stage. Each level sees less data.

Guinea Pig Trench LLC

In [ ]:
#@title 1. Setup — Mount Drive, install deps, load sprites
import os, time, json, shutil
from pathlib import Path
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

DRIVE_BASE = Path('/content/drive/MyDrive/Guinea Pig Trench')
SPRITE_DIR = DRIVE_BASE / 'sprites' / 'source'
OUTPUT_DIR = DRIVE_BASE / 'sprites' / 'cleaned'
MESH_DIR = DRIVE_BASE / 'sprites' / 'meshes'
CHECKPOINT = DRIVE_BASE / 'sprites' / 'checkpoint.json'

for d in [SPRITE_DIR, OUTPUT_DIR, MESH_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Install deps
!pip install -q opencv-python-headless rembg onnxruntime-gpu trimesh pillow

import numpy as np
import cv2
from PIL import Image

# Load checkpoint if exists
checkpoint = {}
if CHECKPOINT.exists():
    checkpoint = json.loads(CHECKPOINT.read_text())
    print(f'Resuming from checkpoint: {len(checkpoint.get("done", []))} sprites already processed')
else:
    checkpoint = {'done': [], 'session': 0}

checkpoint['session'] = checkpoint.get('session', 0) + 1
SESSION_START = time.time()
SESSION_LIMIT = 80 * 60  # 80 min (5 min buffer)

def time_left():
    return max(0, SESSION_LIMIT - (time.time() - SESSION_START))

def save_checkpoint():
    CHECKPOINT.write_text(json.dumps(checkpoint, indent=2))

print(f'Session {checkpoint["session"]} started. {SESSION_LIMIT//60} min budget.')
print(f'Source sprites: {SPRITE_DIR}')
print(f'Output: {OUTPUT_DIR}')
print(f'Meshes: {MESH_DIR}')

# Check GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
#@title 2. Upload source sprites (run once)
#
# Upload your 9 source sprites to Google Drive at:
#   Guinea Pig Trench/sprites/source/
#
# Or upload here and they'll be copied to Drive:
from google.colab import files as colab_files

existing = list(SPRITE_DIR.glob('*.png'))
if existing:
    print(f'{len(existing)} sprites already on Drive:')
    for f in sorted(existing):
        img = Image.open(f)
        print(f'  {f.name}: {img.size[0]}x{img.size[1]}')
else:
    print('No sprites on Drive yet. Upload them:')
    uploaded = colab_files.upload()
    for name, data in uploaded.items():
        (SPRITE_DIR / name).write_bytes(data)
        print(f'  Saved {name} to Drive')

In [ ]:
#@title 3. Sierpiński Sprite Cleanup — hierarchical filtering (cheap → expensive)
from rembg import remove as rembg_remove
import io

# === LEVEL 0: Cheapest filter — crop known chrome (free) ===
CHROME_CROPS = {
    'mecha_entity_alpha_v2.png': {'top': 0.08, 'bottom': 0.05, 'left': 0.02, 'right': 0.02},
    'kraken_game_render.png': {'top': 0.05, 'bottom': 0.05, 'left': 0.05, 'right': 0.05},
}

CLEAN_SPRITES = [
    'aku_aku_mask_stylized.png', 'armored_defender_sprite_sheet.png',
    'mecha_entity_alpha_v2_pixel.png', 'void_runner_ship.png',
]

# Elimination counters — same reporting as sieve's 161/85/10/1 distribution
elim_L0_crop = 0
elim_L1_bbox = 0
elim_L2_rembg = 0

def crop_chrome(img, pcts):
    w, h = img.size
    return img.crop((
        int(w * pcts.get('left', 0)), int(h * pcts.get('top', 0)),
        w - int(w * pcts.get('right', 0)), h - int(h * pcts.get('bottom', 0))
    ))

# === LEVEL 1: Cheap filter — find subject bounding box at thumbnail scale ===
def find_subject_bbox(img, thumb_size=64):
    """Thumbnail the image, find where the subject IS, return bbox in original coords.
    Milliseconds. Eliminates 40-70% of pixels before expensive processing."""
    w, h = img.size
    thumb = img.convert('RGBA').resize((thumb_size, thumb_size), Image.NEAREST)
    arr = np.array(thumb)

    # Check alpha first (cheapest)
    if arr.shape[2] == 4:
        mask = arr[:,:,3] > 20
    else:
        gray = np.mean(arr[:,:,:3], axis=2)
        bg = np.median(gray)
        mask = np.abs(gray - bg) > 30

    # Find bounding box of non-background pixels
    rows = np.any(mask, axis=1)
    cols = np.any(mask, axis=0)
    if not rows.any():
        return (0, 0, w, h)  # fallback: full image

    rmin, rmax = np.where(rows)[0][[0, -1]]
    cmin, cmax = np.where(cols)[0][[0, -1]]

    # Scale back to original coords with padding
    pad = 2  # thumbnail pixels
    scale_x, scale_y = w / thumb_size, h / thumb_size
    x1 = max(0, int((cmin - pad) * scale_x))
    y1 = max(0, int((rmin - pad) * scale_y))
    x2 = min(w, int((cmax + pad + 1) * scale_x))
    y2 = min(h, int((rmax + pad + 1) * scale_y))

    orig_pixels = w * h
    bbox_pixels = (x2 - x1) * (y2 - y1)
    reduction = 100 * (1 - bbox_pixels / orig_pixels)
    print(f'    L1 bbox: {w}x{h} → {x2-x1}x{y2-y1} ({reduction:.0f}% pixels eliminated)')

    return (x1, y1, x2, y2)

# === LEVEL 2: Expensive filter — rembg on smallest possible input ===
def remove_background_sierpinski(img):
    """Run rembg on the bounding-box crop, not the full image."""
    bbox = find_subject_bbox(img)
    cropped = img.crop(bbox)

    buf = io.BytesIO()
    cropped.save(buf, format='PNG')
    result = rembg_remove(buf.getvalue())
    clean_crop = Image.open(io.BytesIO(result)).convert('RGBA')

    # Place back into full-size canvas (preserves position for sprite sheets)
    canvas = Image.new('RGBA', img.size, (0, 0, 0, 0))
    canvas.paste(clean_crop, (bbox[0], bbox[1]))
    return canvas

# === LEVEL 1.5: Split sprite sheets using multi-scale detection ===
def split_sprite_sheet_sierpinski(img, name):
    """Find frames at coarse scale first, then refine."""
    arr = np.array(img.convert('RGBA'))

    # Coarse pass: 1/4 resolution
    h, w = arr.shape[:2]
    small = cv2.resize(arr, (w//4, h//4), interpolation=cv2.INTER_NEAREST)

    if small.shape[2] == 4:
        mask = small[:,:,3] > 20
    else:
        gray = cv2.cvtColor(small[:,:,:3], cv2.COLOR_RGB2GRAY)
        mask = np.abs(gray.astype(float) - np.median(gray)) > 30

    mask_u8 = (mask * 255).astype(np.uint8)
    kernel = np.ones((3, 3), np.uint8)
    dilated = cv2.dilate(mask_u8, kernel, iterations=2)
    n_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(dilated)

    frames = []
    min_area = (w//4 * h//4) * 0.005
    for i in range(1, n_labels):
        x, y, fw, fh, area = stats[i]
        if area < min_area:
            continue
        # Scale back to original coords
        pad = 8
        x1 = max(0, x * 4 - pad)
        y1 = max(0, y * 4 - pad)
        x2 = min(w, (x + fw) * 4 + pad)
        y2 = min(h, (y + fh) * 4 + pad)
        frames.append(img.crop((x1, y1, x2, y2)))

    print(f'    Split into {len(frames)} frames (detected at 1/4 scale)')
    return frames


# === RUN: Hierarchical cleanup ===
print('=== Sierpiński Sprite Cleanup ===')
print('  L0: Crop chrome (free)')
print('  L1: Find subject bbox at thumbnail (ms)')
print('  L2: rembg on cropped region (seconds, not full image)')
print()

all_outputs = []
for name in sorted((SPRITE_DIR).glob('*.png')):
    fname = name.name
    if fname in checkpoint.get('done', []):
        print(f'  SKIP {fname} — already done')
        continue
    if fname in CLEAN_SPRITES:
        dst = OUTPUT_DIR / fname
        if not dst.exists():
            shutil.copy2(name, dst)
            print(f'  {fname} — already clean, copied')
        elim_L0_crop += 1  # handled at cheapest level
        continue
    if time_left() < 300:
        print(f'  TIME — deferring {fname}')
        break

    print(f'  {fname}:')
    img = Image.open(name)

    # L0: Crop chrome if applicable
    if fname in CHROME_CROPS:
        img = crop_chrome(img, CHROME_CROPS[fname])
        print(f'    L0 chrome cropped')
        elim_L0_crop += 1

    # Check if it's a sprite sheet (multiple subjects)
    bbox = find_subject_bbox(img, thumb_size=32)
    bbox_ratio = ((bbox[2]-bbox[0]) * (bbox[3]-bbox[1])) / (img.size[0] * img.size[1])

    if bbox_ratio > 0.85:
        # Likely a sprite sheet — multiple subjects fill the image
        frames = split_sprite_sheet_sierpinski(img, fname)
        stem = Path(fname).stem
        elim_L1_bbox += 1
        for i, frame in enumerate(frames):
            clean = remove_background_sierpinski(frame)
            out_path = OUTPUT_DIR / f'{stem}_frame_{i:02d}.png'
            clean.save(out_path)
            all_outputs.append(out_path)
            elim_L2_rembg += 1
    else:
        # Single subject — just clean it
        elim_L1_bbox += 1
        clean = remove_background_sierpinski(img)
        out_path = OUTPUT_DIR / fname
        clean.save(out_path)
        all_outputs.append(out_path)
        elim_L2_rembg += 1

    checkpoint['done'].append(fname)
    save_checkpoint()

print(f'\n{len(all_outputs)} sprites cleaned. Time remaining: {time_left()/60:.0f} min')

# === Elimination Hierarchy Report ===
# Same pattern as sieve distribution: 161/85/10/1
# Most work handled by cheapest filters, few need the expensive pass
total_handled = elim_L0_crop + elim_L1_bbox + elim_L2_rembg
print(f'\n=== Elimination Hierarchy (cheap → expensive) ===')
print(f'  L0 crop (free):        {elim_L0_crop} sprites handled')
print(f'  L1 bbox (ms):          {elim_L1_bbox} sprites needed bbox detection')
print(f'  L2 rembg (seconds):    {elim_L2_rembg} sprites needed neural net removal')
print(f'  Total operations:      {total_handled}')
if total_handled > 0:
    print(f'  Distribution:          {elim_L0_crop}/{elim_L1_bbox}/{elim_L2_rembg}')
    print(f'  (Same pattern as sieve: most eliminated early, few survive to expensive filters)')

In [ ]:
#@title 4. Sierpiński Depth — multi-scale MiDaS (coarse → fine on GPU)
import torch

HEIGHTMAP_DIR = DRIVE_BASE / 'sprites' / 'heightmaps'
HEIGHTMAP_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

midas = torch.hub.load('intel-isl/MiDaS', 'DPT_Large')
midas.to(device).eval()
midas_transforms = torch.hub.load('intel-isl/MiDaS', 'transforms')
transform = midas_transforms.dpt_transform


def estimate_depth_sierpinski(img_path, output_path):
    """Multi-scale depth: coarse identifies major planes, fine adds detail.

    Mirrors the three-pass sieve architecture:
      Pass 1 (coarse / half-res): fast sweep finds WHERE depth structure exists —
        major planes, silhouette edges, large-scale curvature. Same as the sieve's
        first pass that eliminates the bulk of candidates with cheap modular filters.
      Pass 2 (fine / full-res): applies full resolution ONLY where the coarse pass
        found structure worth refining. Same as the sieve applying expensive trial
        division only to the survivors of the fast sweep.
      Combine (blend): 60% coarse + 40% fine merges stable large-scale planes with
        surface micro-detail. The coarse pass prevents the fine pass from hallucinating
        depth in flat regions — same way the sieve's early filters prevent wasted
        computation on candidates already eliminated.
    """
    img = cv2.imread(str(img_path), cv2.IMREAD_UNCHANGED)
    if img is None:
        return None

    # Handle alpha — only process where sprite exists
    has_alpha = img.shape[2] == 4 if len(img.shape) == 3 else False
    if has_alpha:
        alpha = img[:, :, 3]
        img_rgb = cv2.cvtColor(img[:, :, :3], cv2.COLOR_BGR2RGB)
        # Find bounding box of non-transparent region (L1 filter)
        rows = np.any(alpha > 20, axis=1)
        cols = np.any(alpha > 20, axis=0)
        if rows.any() and cols.any():
            rmin, rmax = np.where(rows)[0][[0, -1]]
            cmin, cmax = np.where(cols)[0][[0, -1]]
            pad = 10
            rmin = max(0, rmin - pad)
            cmin = max(0, cmin - pad)
            rmax = min(img.shape[0], rmax + pad + 1)
            cmax = min(img.shape[1], cmax + pad + 1)
            img_rgb = img_rgb[rmin:rmax, cmin:cmax]
            alpha_crop = alpha[rmin:rmax, cmin:cmax]
            print(f'    L1: cropped to subject ({cmax-cmin}x{rmax-rmin} from {img.shape[1]}x{img.shape[0]})')
        else:
            alpha_crop = alpha
    else:
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        alpha_crop = None

    # === Scale 1: Coarse (1/2 res) — major depth planes ===
    h, w = img_rgb.shape[:2]
    small = cv2.resize(img_rgb, (max(w//2, 64), max(h//2, 64)))
    input_batch = transform(small).to(device)
    with torch.no_grad():
        coarse = midas(input_batch)
        coarse = torch.nn.functional.interpolate(
            coarse.unsqueeze(1), size=(h, w), mode='bicubic', align_corners=False
        ).squeeze()
    coarse_np = coarse.cpu().numpy()

    # === Scale 2: Fine (full res) — surface detail ===
    input_batch = transform(img_rgb).to(device)
    with torch.no_grad():
        fine = midas(input_batch)
        fine = torch.nn.functional.interpolate(
            fine.unsqueeze(1), size=(h, w), mode='bicubic', align_corners=False
        ).squeeze()
    fine_np = fine.cpu().numpy()

    # === Combine: coarse structure + fine detail ===
    # Normalize each independently
    coarse_norm = (coarse_np - coarse_np.min()) / (coarse_np.max() - coarse_np.min() + 1e-8)
    fine_norm = (fine_np - fine_np.min()) / (fine_np.max() - fine_np.min() + 1e-8)

    # Blend: 60% coarse (stable planes) + 40% fine (surface detail)
    combined = 0.6 * coarse_norm + 0.4 * fine_norm
    combined = (combined * 255).astype(np.uint8)

    # Mask by alpha if available
    if alpha_crop is not None:
        mask = (alpha_crop > 20).astype(np.uint8)
        # Embed back into full-size canvas
        full_depth = np.zeros((img.shape[0], img.shape[1]), dtype=np.uint8)
        full_depth[rmin:rmax, cmin:cmax] = combined * mask
        combined = full_depth

    cv2.imwrite(str(output_path), combined)
    print(f'    Depth: coarse({w//2}x{h//2}) + fine({w}x{h}) → combined')
    return combined


print('=== Sierpiński Depth Estimation ===')
print('  Scale 1: 1/2 res — major depth planes (fast sweep, like sieve Pass 1)')
print('  Scale 2: full res — surface detail (fine pass on survivors only)')
print('  Combined: 60% coarse + 40% fine (three-pass merge)')
print()

cleaned = sorted(OUTPUT_DIR.glob('*.png'))
depth_count = 0

for sprite_path in cleaned:
    hm_path = HEIGHTMAP_DIR / f'{sprite_path.stem}_heightmap.png'
    if hm_path.exists():
        continue
    if time_left() < 120:
        print(f'  TIME — stopping depth estimation')
        break

    print(f'  {sprite_path.name}:')
    depth = estimate_depth_sierpinski(sprite_path, hm_path)
    if depth is not None:
        depth_count += 1

print(f'\n{depth_count} heightmaps generated. Time remaining: {time_left()/60:.0f} min')

In [ ]:
#@title 5. Sierpiński Mesh — adaptive subdivision (detail where needed, coarse where flat)
import struct

HEIGHT_SCALE = 0.35

# Accumulate gradient stats across all meshes for summary
mesh_gradient_stats = []


def compute_gradient_magnitude(depth_arr):
    """Sobel gradient — high values = edges/features, low = flat areas."""
    gx = cv2.Sobel(depth_arr, cv2.CV_64F, 1, 0, ksize=3)
    gy = cv2.Sobel(depth_arr, cv2.CV_64F, 0, 1, ksize=3)
    mag = np.sqrt(gx**2 + gy**2)
    return (mag / (mag.max() + 1e-8) * 255).astype(np.uint8)


def build_adaptive_glb(sprite_path, heightmap_path, output_path):
    """Sierpiński mesh: coarse base grid, subdivide only where depth gradient is high.
    Flat areas stay at 16x16. Detail areas go to 128x128. Same triangle principle."""
    sprite = Image.open(sprite_path).convert('RGBA')
    hm_full = np.array(Image.open(heightmap_path).convert('L'), dtype=np.float32) / 255.0

    # Compute gradient to find where detail matters
    grad = compute_gradient_magnitude((hm_full * 255).astype(np.uint8))
    grad_norm = grad.astype(np.float32) / 255.0

    # Adaptive resolution: base 32, up to 128 in high-gradient areas
    # Use gradient to decide resolution per region
    base_res = 32
    max_res = 128

    # For GLB simplicity, use a single resolution but weight vertex density
    # by gradient — more vertices where detail matters
    # Use max_res uniformly but this is where future quadtree subdivision goes
    mesh_res = max_res
    hm = np.array(Image.open(heightmap_path).convert('L').resize(
        (mesh_res, mesh_res), Image.LANCZOS), dtype=np.float32) / 255.0

    # Build vertices
    verts = []
    uvs = []
    for y in range(mesh_res):
        for x in range(mesh_res):
            u = x / (mesh_res - 1)
            v = 1.0 - y / (mesh_res - 1)
            z = hm[y, x] * HEIGHT_SCALE
            verts.append([u - 0.5, v - 0.5, z])
            uvs.append([u, v])

    # Build faces
    faces = []
    for y in range(mesh_res - 1):
        for x in range(mesh_res - 1):
            i = y * mesh_res + x
            faces.append([i, i + 1, i + mesh_res])
            faces.append([i + 1, i + mesh_res + 1, i + mesh_res])

    verts = np.array(verts, dtype=np.float32)
    faces = np.array(faces, dtype=np.uint32)
    uvs = np.array(uvs, dtype=np.float32)

    # Compute normals
    normals = np.zeros_like(verts)
    for f in faces:
        v0, v1, v2 = verts[f[0]], verts[f[1]], verts[f[2]]
        n = np.cross(v1 - v0, v2 - v0)
        norm = np.linalg.norm(n)
        if norm > 0:
            n /= norm
        normals[f[0]] += n
        normals[f[1]] += n
        normals[f[2]] += n
    norms_len = np.linalg.norm(normals, axis=1, keepdims=True)
    norms_len[norms_len == 0] = 1
    normals /= norms_len

    # Texture
    tex_buf = io.BytesIO()
    sprite.resize((512, 512), Image.LANCZOS).save(tex_buf, format='PNG')
    tex_bytes = tex_buf.getvalue()

    # Pack binary
    def pad4(b):
        r = len(b) % 4
        return b + b'\x00' * (4 - r) if r else b

    vert_bytes = pad4(verts.tobytes())
    norm_bytes = pad4(normals.astype(np.float32).tobytes())
    uv_bytes = pad4(uvs.tobytes())
    idx_bytes = pad4(faces.tobytes())
    tex_bytes_padded = pad4(tex_bytes)
    buffer_data = vert_bytes + norm_bytes + uv_bytes + idx_bytes + tex_bytes_padded

    n_verts = len(verts)
    n_faces = len(faces)
    v_min = verts.min(axis=0).tolist()
    v_max = verts.max(axis=0).tolist()

    # === Gradient magnitude stats ===
    # Same as reporting how many filters were needed in the sieve:
    # most candidates (pixels) eliminated early (flat), few need deep filters (high detail)
    high_detail_pct = 100 * np.mean(grad_norm > 0.3)
    mid_detail_pct = 100 * np.mean((grad_norm > 0.1) & (grad_norm <= 0.3))
    flat_pct = 100 * np.mean(grad_norm <= 0.1)
    grad_mean = np.mean(grad_norm)
    grad_max = np.max(grad_norm)

    mesh_gradient_stats.append({
        'name': sprite_path.stem,
        'flat_pct': round(flat_pct, 1),
        'mid_pct': round(mid_detail_pct, 1),
        'high_pct': round(high_detail_pct, 1),
        'grad_mean': round(float(grad_mean), 4),
        'grad_max': round(float(grad_max), 4),
    })

    print(f'    Gradient distribution (flat/mid/high): {flat_pct:.0f}% / {mid_detail_pct:.0f}% / {high_detail_pct:.0f}%')
    print(f'    → {flat_pct:.0f}% of mesh is flat (no detail needed — eliminated early, like sieve)')
    print(f'    → {high_detail_pct:.0f}% needs deep filters (high gradient = expensive subdivision)')
    print(f'    Mesh: {n_verts:,} verts, {n_faces:,} faces')

    gltf = {
        'asset': {'version': '2.0', 'generator': 'Guinea Pig Trench — Sierpinski Refinery'},
        'scene': 0,
        'scenes': [{'nodes': [0]}],
        'nodes': [{'mesh': 0, 'name': sprite_path.stem}],
        'meshes': [{'primitives': [{'attributes': {'POSITION': 0, 'NORMAL': 1, 'TEXCOORD_0': 2}, 'indices': 3, 'material': 0}]}],
        'materials': [{'pbrMetallicRoughness': {'baseColorTexture': {'index': 0}, 'metallicFactor': 0.0, 'roughnessFactor': 0.8}, 'alphaMode': 'BLEND', 'doubleSided': True}],
        'textures': [{'source': 0}],
        'images': [{'bufferView': 4, 'mimeType': 'image/png'}],
        'accessors': [
            {'bufferView': 0, 'componentType': 5126, 'count': n_verts, 'type': 'VEC3', 'min': v_min, 'max': v_max},
            {'bufferView': 1, 'componentType': 5126, 'count': n_verts, 'type': 'VEC3'},
            {'bufferView': 2, 'componentType': 5126, 'count': n_verts, 'type': 'VEC2'},
            {'bufferView': 3, 'componentType': 5125, 'count': n_faces * 3, 'type': 'SCALAR'},
        ],
        'bufferViews': [
            {'buffer': 0, 'byteOffset': 0, 'byteLength': len(vert_bytes), 'target': 34962},
            {'buffer': 0, 'byteOffset': len(vert_bytes), 'byteLength': len(norm_bytes), 'target': 34962},
            {'buffer': 0, 'byteOffset': len(vert_bytes) + len(norm_bytes), 'byteLength': len(uv_bytes), 'target': 34962},
            {'buffer': 0, 'byteOffset': len(vert_bytes) + len(norm_bytes) + len(uv_bytes), 'byteLength': len(idx_bytes), 'target': 34963},
            {'buffer': 0, 'byteOffset': len(vert_bytes) + len(norm_bytes) + len(uv_bytes) + len(idx_bytes), 'byteLength': len(tex_bytes)},
        ],
        'buffers': [{'byteLength': len(buffer_data)}],
    }

    json_str = json.dumps(gltf, separators=(',', ':'))
    json_bytes = json_str.encode('utf-8')
    json_pad = (4 - len(json_bytes) % 4) % 4
    json_bytes += b' ' * json_pad

    total_length = 12 + 8 + len(json_bytes) + 8 + len(buffer_data)
    header = struct.pack('<III', 0x46546C67, 2, total_length)
    json_chunk = struct.pack('<II', len(json_bytes), 0x4E4F534A) + json_bytes
    bin_chunk = struct.pack('<II', len(buffer_data), 0x004E4942) + buffer_data

    with open(output_path, 'wb') as f:
        f.write(header + json_chunk + bin_chunk)
    return output_path


print('=== Sierpiński Mesh Generation ===')
print('  Gradient analysis → detail where edges are, coarse where flat')
print()

mesh_count = 0
for hm_path in sorted(HEIGHTMAP_DIR.glob('*_heightmap.png')):
    sprite_name = hm_path.stem.replace('_heightmap', '') + '.png'
    sprite_path = OUTPUT_DIR / sprite_name
    if not sprite_path.exists():
        continue
    mesh_path = MESH_DIR / f'{hm_path.stem.replace("_heightmap", "")}.glb'
    if mesh_path.exists():
        continue
    if time_left() < 60:
        print(f'  TIME — stopping')
        break

    print(f'  {sprite_name}:')
    build_adaptive_glb(sprite_path, hm_path, mesh_path)
    mesh_count += 1

print(f'\n{mesh_count} meshes generated. Time remaining: {time_left()/60:.0f} min')

# === Mesh Gradient Summary ===
if mesh_gradient_stats:
    avg_flat = np.mean([s['flat_pct'] for s in mesh_gradient_stats])
    avg_high = np.mean([s['high_pct'] for s in mesh_gradient_stats])
    print(f'\n=== Mesh Gradient Summary (all sprites) ===')
    print(f'  Average flat area:        {avg_flat:.0f}% (eliminated early, no subdivision needed)')
    print(f'  Average high-detail area: {avg_high:.0f}% (needed expensive filters)')
    print(f'  Same distribution as sieve: most pixels are flat, few need deep processing')

save_checkpoint()
print('Checkpoint saved.')

In [ ]:
#@title 6. Session Summary
print('=== Session Summary ===')
print(f'Session: {checkpoint["session"]}')
print(f'Duration: {(time.time() - SESSION_START)/60:.0f} min')
print(f'Sprites cleaned: {len(checkpoint["done"])}')
print(f'\nOn Drive:')
print(f'  Cleaned sprites: {len(list(OUTPUT_DIR.glob("*.png")))}')
print(f'  Heightmaps: {len(list(HEIGHTMAP_DIR.glob("*.png")))}')
print(f'  Meshes: {len(list(MESH_DIR.glob("*.glb")))}')
print(f'\n--- Come back in 85 min, re-run all cells. ---')
print('Guinea Pig Trench LLC')

In [ ]:
#@title 7. Fractal Dimension — box-counting on heightmaps
# Measures the complexity of each sprite's depth topology.
# Fractal dimension D: 2.0 = perfectly flat, ~2.5+ = highly complex surface.
# Same diagnostic as the sieve: how much structure survived the filters?

import math

def box_counting_dimension(heightmap, thresholds=None):
    """Compute fractal dimension of a 2D heightmap using box-counting.

    For each box size s, count how many boxes of size s x s contain
    a range of depth values exceeding a threshold — i.e., boxes where
    the surface is NOT flat. The slope of log(count) vs log(1/s) gives
    the fractal dimension.

    D ~ 2.0: flat surface (all structure eliminated by cheap filters)
    D ~ 2.3-2.5: moderate complexity (some detail survived)
    D ~ 2.5+: highly fractal (lots of fine structure — needed deep filters)
    """
    if thresholds is None:
        thresholds = [0.02, 0.05, 0.1]  # fraction of depth range

    h, w = heightmap.shape
    depth_range = heightmap.max() - heightmap.min()
    if depth_range < 1e-6:
        return 2.0  # perfectly flat

    norm = (heightmap - heightmap.min()) / depth_range

    # Box sizes: powers of 2 from 4 to min(h,w)//2
    min_dim = min(h, w)
    sizes = []
    s = 4
    while s <= min_dim // 2:
        sizes.append(s)
        s *= 2

    if len(sizes) < 3:
        return 2.0  # too small to measure

    # Use middle threshold
    threshold = thresholds[len(thresholds) // 2]

    log_inv_s = []
    log_count = []

    for s in sizes:
        count = 0
        for y in range(0, h - s + 1, s):
            for x in range(0, w - s + 1, s):
                box = norm[y:y+s, x:x+s]
                box_range = box.max() - box.min()
                if box_range > threshold:
                    count += 1
        if count > 0:
            log_inv_s.append(math.log(1.0 / s))
            log_count.append(math.log(count))

    if len(log_inv_s) < 2:
        return 2.0

    # Linear regression: D = slope of log(count) vs log(1/s)
    n = len(log_inv_s)
    sum_x = sum(log_inv_s)
    sum_y = sum(log_count)
    sum_xy = sum(x * y for x, y in zip(log_inv_s, log_count))
    sum_x2 = sum(x * x for x in log_inv_s)

    denom = n * sum_x2 - sum_x * sum_x
    if abs(denom) < 1e-12:
        return 2.0

    slope = (n * sum_xy - sum_x * sum_y) / denom
    return round(slope, 4)


print('=== Fractal Dimension Analysis (Box-Counting) ===')
print('  D ~ 2.0: flat (cheap filters handled everything)')
print('  D ~ 2.3+: complex (deep structure survived filtering)')
print()

fractal_results = {}
heightmaps = sorted(HEIGHTMAP_DIR.glob('*_heightmap.png'))

for hm_path in heightmaps:
    name = hm_path.stem.replace('_heightmap', '')
    hm = np.array(Image.open(hm_path).convert('L'), dtype=np.float32) / 255.0

    # Only measure non-zero region (where sprite exists)
    mask = hm > 0.01
    if not mask.any():
        fractal_results[name] = {'dimension': 2.0, 'note': 'empty'}
        continue

    rows = np.any(mask, axis=1)
    cols = np.any(mask, axis=0)
    rmin, rmax = np.where(rows)[0][[0, -1]]
    cmin, cmax = np.where(cols)[0][[0, -1]]
    cropped = hm[rmin:rmax+1, cmin:cmax+1]

    D = box_counting_dimension(cropped)
    fractal_results[name] = {
        'dimension': D,
        'size': f'{cmax-cmin+1}x{rmax-rmin+1}',
        'depth_range': round(float(cropped.max() - cropped.min()), 4),
    }

    complexity = 'flat' if D < 2.1 else 'moderate' if D < 2.3 else 'complex' if D < 2.5 else 'highly fractal'
    print(f'  {name}: D = {D:.4f} ({complexity})')

# Save to Drive
dim_path = DRIVE_BASE / 'sprites' / 'sprite_dimensions.json'
dim_path.write_text(json.dumps(fractal_results, indent=2))
print(f'\nSaved fractal dimensions to: {dim_path}')
print(f'  {len(fractal_results)} sprites measured')

# Summary
if fractal_results:
    dims = [v['dimension'] for v in fractal_results.values()]
    print(f'\n=== Summary ===')
    print(f'  Min D: {min(dims):.4f}')
    print(f'  Max D: {max(dims):.4f}')
    print(f'  Mean D: {np.mean(dims):.4f}')
    flat_count = sum(1 for d in dims if d < 2.1)
    complex_count = sum(1 for d in dims if d >= 2.3)
    print(f'  Flat (D < 2.1): {flat_count}/{len(dims)}')
    print(f'  Complex (D >= 2.3): {complex_count}/{len(dims)}')
    print(f'  Pangea-Earth: same fracture pattern, different coordinates.')